# 04. v1 — 표 구조 반영

v0은 표를 숫자 나열로 읽어서, 어느 숫자가 어느 열(연도·분기·연령)인지 알 수 없었습니다.
v1은 표를 **"행 이름: 열 제목 = 값"** 형태의 글로 바꾸고, 표 제목과 **섹션 제목**을 앞에 붙입니다.
(표 재구성 코드: `app/ingest/tables.py`)

**공정한 비교를 위해 그대로 두는 것:** 답변 프롬프트, 답변 모델, TOP_K=5, 본문 청크 크기 800/100, 채점기, 테스트셋.
바뀌는 것은 "문서를 읽어서 청크로 만드는 방법" 하나뿐입니다.

- **API 사용 (요금 발생)** 칸: ④ 인덱스 만들기 (약 $0.006), ⑥ 평가 (약 $0.08)
- 두 칸 모두 결과 파일이 있으면 **불러오기만** 합니다. (다시 실행해도 요금 없음)

## ① 설정

In [1]:
import os
import re
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*langchain-community.*")

from dotenv import load_dotenv
import pandas as pd
import pymupdf
from langchain_core.documents import Document
from langchain_core.callbacks import get_usage_metadata_callback
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_DIR / ".env")
sys.path.insert(0, str(PROJECT_DIR))  # app/, eval/ 폴더의 코드를 불러오기 위해

from app.ingest.tables import update_sections, table_lines, text_without_tables, split_table
from eval.rag_eval import load_testset, run_eval, summarize, JudgeResult

LLM_MODEL = os.getenv("LLM_MODEL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
JUDGE_MODEL = os.getenv("JUDGE_MODEL")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", 800))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", 100))
TOP_K = 5

PDF_DIR = PROJECT_DIR / "data" / "pdf"
INDEX_DIR = PROJECT_DIR / "data" / "index" / "v1_faiss"
RESULTS_DIR = PROJECT_DIR / "eval" / "results"

DOCS = {
    "D01": "2026_youth_employment_survey.pdf",
    "D02": "2025_content_industry_trend.pdf",
    "D03": "2025_social_survey.pdf",
}
# 01_pdf_explore에서 정한 사용 범위 (v0과 같음)
PAGE_RANGES = {"D01": [(1, 40)], "D02": [(13, 22), (37, 54)], "D03": [(1, 56)]}

def pages_in_range(doc_id, page_no):
    return any(a <= page_no <= b for a, b in PAGE_RANGES[doc_id])

print("API 키:", "있음" if os.getenv("OPENAI_API_KEY") else "없음 (.env 확인)")
print("답변 모델:", LLM_MODEL, "| 채점 모델:", JUDGE_MODEL, "| 임베딩:", EMBEDDING_MODEL, "| TOP_K:", TOP_K)

API 키: 있음
답변 모델: gpt-6-luna | 채점 모델: gpt-6.1-terra | 임베딩: text-embedding-3-small | TOP_K: 5


## ② 문서 읽기 — 표는 표대로, 본문은 표를 빼고
쪽마다 다음을 합니다.
1. 쪽 위쪽의 큰 제목(Ⅰ. / 1. / 제1장)을 읽어 **현재 섹션**을 갱신합니다. 사용 범위 밖의 쪽도 읽어야 섹션이 끊기지 않습니다.
2. PyMuPDF로 **표 위치**를 찾고, `table_lines`로 표를 "행 이름: 열 제목 = 값" 글로 바꿉니다.
3. 재구성에 성공한 표의 영역은 **본문에서 뺍니다.** (같은 숫자가 뒤섞인 채로 본문에 또 들어가는 것을 막음)

In [2]:
def clean(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

text_docs, table_docs = [], []
n_found = n_parsed = 0
for doc_id, filename in DOCS.items():
    pdf = pymupdf.open(PDF_DIR / filename)
    sections = []
    for page in pdf:
        page_no = page.number + 1
        section = update_sections(page, sections)
        if not pages_in_range(doc_id, page_no):
            continue
        bboxes = []
        for t in page.find_tables().tables:
            n_found += 1
            result = table_lines(page, t.bbox, section)
            if result is None:          # 숫자 표가 아니면(글 상자 등) 본문으로 둠
                continue
            n_parsed += 1
            bboxes.append(t.bbox)
            for chunk in split_table(*result, max_chars=CHUNK_SIZE):
                table_docs.append(Document(page_content=chunk,
                                           metadata={"doc_id": doc_id, "page": page_no, "kind": "table"}))
        text = clean(text_without_tables(page, bboxes))
        if len(text) >= 50:
            text_docs.append(Document(page_content=text,
                                      metadata={"doc_id": doc_id, "page": page_no, "kind": "text"}))

print(f"표: 찾은 것 {n_found}개 중 재구성 {n_parsed}개 → 표 조각 {len(table_docs)}개")
print("본문 쪽 수:", len(text_docs))

표: 찾은 것 131개 중 재구성 127개 → 표 조각 360개
본문 쪽 수: 121


## ③ 청크
본문은 v0과 같은 방식(800자, 겹침 100자)으로 나누고, 표 조각은 ②에서 이미 800자 이하로 나눴으므로 그대로 씁니다.

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
text_chunks = [c for c in splitter.split_documents(text_docs) if len(c.page_content) >= 30]
chunks = text_chunks + table_docs

total_chars = sum(len(c.page_content) for c in chunks)
print(f"본문 청크 {len(text_chunks)}개 + 표 청크 {len(table_docs)}개 = {len(chunks)}개")
print(f"전체 글자 수 {total_chars:,} (임베딩 예상 비용 약 ${total_chars / 1.5 / 1e6 * 0.02:.4f})")

본문 청크 152개 + 표 청크 360개 = 512개
전체 글자 수 309,209 (임베딩 예상 비용 약 $0.0041)


## ④ 인덱스 만들기 — API 사용 (요금 발생, 약 $0.006)
`data/index/v1_faiss`가 이미 있으면 불러오기만 합니다.

In [4]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
if INDEX_DIR.exists():
    vectorstore = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)  # 직접 만든 파일
    print("불러옴:", INDEX_DIR)
else:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(str(INDEX_DIR))
    print("만들고 저장:", INDEX_DIR)
print("인덱스 청크 수:", vectorstore.index.ntotal)

만들고 저장: /home/user/git_project/project6/ReportQA/data/index/v1_faiss
인덱스 청크 수: 512


## ⑤ 표 청크 확인
v0에서 틀렸던 두 종류를 봅니다.
- D02 45쪽 콘텐츠산업 매출액 표: 분기 열 제목이 숫자에 붙었는지
- D01 28쪽 통계표: "15~34세" 섹션 제목이 붙었는지 (v0은 15~29세 표와 헷갈렸음)

In [5]:
def show(doc_id, page_no, keyword, n_lines=3):
    for c in table_docs:
        m = c.metadata
        if m["doc_id"] == doc_id and m["page"] == page_no and keyword in c.page_content:
            lines = c.page_content.splitlines()
            print(f"===== {doc_id} p.{page_no}")
            print(lines[0])
            print("\n".join([l for l in lines[1:] if keyword in l][:n_lines]))
            print()
            return

show("D02", 45, "음악")
show("D01", 28, "휴학")
show("D01", 14, "3년제")

===== D02 p.45
[표] [제장1 2025년 4분기 및 연간 콘텐츠산업 규모 > 2. 콘텐츠산업 규모 분석 결과] 표 1-1 ’24년 1분기 ~ ’25년 4분기 콘텐츠산업 매출액 규모 (단위 : 백만 원)
- 음악: 2024 년 P 1 분기 = 2,507,287; 2024 년 P 2 분기 = 3,046,733; 2024 년 P 3 분기 = 3,400,830; 2024 년 P 4 분기 = 4,318,044; 2025 년 P 1 분기 = 3,540,869; 2025 년 P 2 분기 = 3,755,533; 2025 년 P 3 분기 = 4,015,543; 2025 년 P 4 분기 = 4,056,713; 2025 년 P = (9.0%); 증감률 전분기 대비 = 1.0%; 증감률 전년 동기 대비 = 6.1%

===== D01 p.28
[표] [Ⅲ. 2026년 5월 청년층 부가조사 통계표 > 1. 2026년 5월 청년층(15~34세) 부가조사 통계표] (단위: 천명, %, %p)
- 휴학 경험 있음: 2025. 5 15-34세 대졸자 = 2,521; 2025. 5 남자 = 1,737; 2025. 5 여자 = 785; 2026. 5 15-34세 대졸자 = 2,512; 2026. 5 남자 = 1,727; 2026. 5 15-34세 여자 = 785; 증감 15-34세 대졸자 = -9; 증감 남자 = -10; 증감 여자 = 1
- 휴학 경험 있음: 2025. 5 15-34세 대졸자 = (48.7); 2025. 5 남자 = (74.7); 2025. 5 여자 = (27.5); 2026. 5 15-34세 대졸자 = (49.8); 2026. 5 남자 = (75.2); 2026. 5 15-34세 여자 = (28.6); 증감 15-34세 대졸자 = (1.1); 증감 남자 = (0.5); 증감 여자 = (1.1)

===== D01 p.14
[표] [Ⅱ. 2026년 5월 청년층(15~29세) 부가조사 결과 > 2. 대학 졸업 소요기간 및 휴학상태] < 대졸자의 휴학경험 > (단위:

## ⑥ 평가 — API 사용 (요금 발생, 약 $0.08)
v0과 **같은 프롬프트**로 답하고, 같은 채점기로 채점해 `eval/results/v1.csv`에 저장합니다. 파일이 있으면 불러옵니다.

In [6]:
llm = ChatOpenAI(model=LLM_MODEL)

# 02·03 노트북과 같은 프롬프트 (버전 간 비교를 위해 바꾸지 않음)
PROMPT = """당신은 통계 보고서 내용으로만 답하는 도우미입니다.
규칙:
1. 아래 [근거]에 있는 내용으로만 답하세요. 추측하지 마세요.
2. 답의 끝에 사용한 근거를 [문서ID p.쪽] 형식으로 표시하세요. 예: [D01 p.8]
3. [근거]에서 답을 찾을 수 없으면 정확히 "모름"이라고만 답하세요.

[근거]
{context}

[질문]
{question}"""

def answer_v1(question):
    found = vectorstore.similarity_search(question, k=TOP_K)
    context = "\n\n".join(f"[{d.metadata['doc_id']} p.{d.metadata['page']}]\n{d.page_content}" for d in found)
    response = llm.invoke(PROMPT.format(context=context, question=question))
    return response.text, [(d.metadata["doc_id"], d.metadata["page"]) for d in found]

testset = load_testset(PROJECT_DIR / "eval" / "testset.csv")
RESULT_PATH = RESULTS_DIR / "v1.csv"
if RESULT_PATH.exists():
    results = pd.read_csv(RESULT_PATH)
    print("불러옴:", RESULT_PATH)
else:
    judge_llm = ChatOpenAI(model=JUDGE_MODEL).with_structured_output(JudgeResult)
    with get_usage_metadata_callback() as usage:
        results = run_eval(testset, answer_v1, judge_llm, k=TOP_K)
    results.to_csv(RESULT_PATH, index=False)
    print("저장:", RESULT_PATH)
    for model, u in usage.usage_metadata.items():
        print(f"{model}: 입력 {u['input_tokens']:,} / 출력 {u['output_tokens']:,} 토큰")

10/110 완료


20/110 완료


30/110 완료


40/110 완료


50/110 완료


60/110 완료


70/110 완료


80/110 완료


90/110 완료


100/110 완료


110/110 완료
저장: /home/user/git_project/project6/ReportQA/eval/results/v1.csv
gpt-6-luna: 입력 241,417 / 출력 13,811 토큰
gpt-6.1-terra: 입력 25,982 / 출력 2,930 토큰


## ⑦ v0 vs v1 비교
같은 시험지·같은 채점기이므로 점수 차이는 "문서를 읽는 방법"의 차이입니다.

In [7]:
v0 = pd.read_csv(RESULTS_DIR / "v0.csv")
compare = pd.DataFrame({"v0": summarize(v0), "v1": summarize(results)})
compare["변화(%p)"] = ((compare["v1"] - compare["v0"]) * 100).round(1)
print("판정 분포 v1:", results["judge"].value_counts().to_dict())
compare

판정 분포 v1: {'correct': 81, 'refused': 21, 'wrong': 8}


,v0,v1,변화(%p)
Recall@5,0.870,0.890,2.0
정답률,0.673,0.736,6.3
텍스트,0.925,0.950,2.5
표,0.533,0.833,30.0
차트(vector),0.611,0.444,-16.7
차트(image),0.000,0.000,0.0
환각률,0.000,0.000,0.0


In [8]:
pd.set_option("display.max_colwidth", 70)
both = (v0[["id", "type", "judge", "recall_hit"]]
        .merge(results[["id", "judge", "recall_hit", "answer_text"]], on="id", suffixes=("_v0", "_v1"))
        .merge(testset[["id", "question", "answer"]], on="id"))
improved = both[(both["judge_v0"] != "correct") & (both["judge_v1"] == "correct")]
regressed = both[(both["judge_v0"] == "correct") & (both["judge_v1"] != "correct")]
print(f"좋아진 문항 {len(improved)}개 / 나빠진 문항 {len(regressed)}개")
print("\n[좋아진 문항]")
display(improved[["id", "type", "question", "recall_hit_v0", "recall_hit_v1"]])
print("\n[나빠진 문항]")
display(regressed[["id", "type", "question", "answer", "answer_text", "recall_hit_v0", "recall_hit_v1"]])

좋아진 문항 15개 / 나빠진 문항 8개

[좋아진 문항]


,id,type,question,recall_hit_v0,recall_hit_v1
12,T013,text,2026년 5월 여자 대졸자의 휴학사유 중 가장 높은 것은?,True,True
15,B003,table,2026년 5월 20~24세 휴학생 수는?,False,True
18,B006,table,2026년 5월 3년제 이하 남자 대졸자의 휴학경험 비율은?,True,True
20,B008,table,2026년 5월 여자 취업경험자의 첫 일자리 중 교육 서비스업 비율은?,False,True
21,B009,table,2026년 5월 첫 일자리 취업 당시 임금이 300만원 이상인 비율은?,True,True
45,T024,text,분기별 콘텐츠산업 동향분석의 표본 사업체 수는?,True,True
49,B011,table,2025년 4분기 음악 산업 매출액은?,True,True
52,B014,table,2024년 4분기 광고 산업 매출액은?,True,True
53,B015,table,2025년 3분기 캐릭터 산업 매출액은?,True,True
55,B017,table,2023년 연간 게임 산업 수출액은?,False,True



[나빠진 문항]


,id,type,question,answer,answer_text,recall_hit_v0,recall_hit_v1
17,B005,table,2026년 5월 3년제 이하 여자 대졸자의 평균 졸업 소요기간은?,2년 6.3개월,모름,True,True
22,B010,table,2026년 5월 취업시험 준비 분야 중 교원임용 비율은?,5.1%,모름,True,False
35,T014,text,2025년 4분기 콘텐츠산업 수출액은 전년 동기 대비 몇 % 변했나?,9.7% 감소,9.7% 증가했습니다. [D02 p.47],True,True
82,C013,chart,2015년 건강보험료 납부에 부담을 느낀다는 응답 비율은?,64.5%,60.8%입니다. [D03 p.16],True,True
88,C019,chart,2019년 장애인과 지속적인 유대관계가 있다는 응답 비율은?,17.9%,모름,True,True
90,C021,chart,2019년 자신의 사회·경제적 지위를 '중'이라고 생각하는 비율은?,58.5%,57.9% [D03 p.38],True,True
93,C024,chart,2017년 독서 인구 1인당 평균 독서 권수는?,17.3권,2017년 독서 인구 1인당 평균 독서 권수는 16.5권입니다. [D03 p.43],True,True
98,C029,chart,2015년 고용(실직·이직)에 대해 불안함을 느낀다는 응답 비율은?,61.0%,2015년에는 39.0%였습니다. [D03 p.53],True,True


In [9]:
# v1에서도 틀린 표 문항: 검색 실패인지, 찾았는데 못 읽은 것인지
still = both[(both["type"] == "table") & (both["judge_v1"] != "correct")]
print("v1에서도 틀린 표 문항:", len(still), "개 | 검색 성공:", int(still["recall_hit_v1"].sum()))
still[["id", "question", "answer", "answer_text", "recall_hit_v1"]]

v1에서도 틀린 표 문항: 5 개 | 검색 성공: 3


,id,question,answer,answer_text,recall_hit_v1
14,B002,2026년 5월 25~29세 경제활동참가율은?,76.9%,모름,True
16,B004,2026년 5월 청년층 졸업 취업자 중 정보통신업 취업자 수는?,183천명,모름,True
17,B005,2026년 5월 3년제 이하 여자 대졸자의 평균 졸업 소요기간은?,2년 6.3개월,모름,True
22,B010,2026년 5월 취업시험 준비 분야 중 교원임용 비율은?,5.1%,모름,False
61,B023,2024년 4분기 콘텐츠산업 상장사 매출액은?,"168,407억 원",모름,False


## ⑧ 발견한 것

### 1. 점수 변화
v1은 두 번 실행했습니다. 처음 결과를 보고 표 재구성 오류 두 가지를 고친 뒤 다시 평가했고, **이 노트북의 출력은 보완 후 결과**입니다.

| 지표 | v0 | v1 (처음) | **v1 (보완 후)** |
|---|---|---|---|
| 정답률 | 67.3% | 71.8% | **73.6%** |
| 표 | 53.3% | 76.7% | **83.3%** |
| 텍스트 | 92.5% | 92.5% | 95.0% |
| 차트(벡터) | 61.1% | 50.0% | 44.4% |
| 차트(이미지) | 0% | 0% | 0% |
| Recall@5 | 87.0% | 89.0% | 89.0% |
| 환각률 | 0% | 0% | 0% |

**표 정답률이 53.3% → 83.3% (+30%p)로 올랐습니다.** v1의 목표를 달성했습니다.
- D02 콘텐츠산업 매출·수출 표(B011·B014·B015·B017·B019): 분기 열 제목이 숫자에 붙어 정확히 읽음
- D01 15~29세 / 15~34세 혼동(B006·B009·T013): 섹션 제목 `[Ⅲ. … 통계표 > 1. … (15~34세) …]`으로 구분됨
- 검색도 나아짐(B003·B008·B017·B022는 v0에서 정답 쪽을 못 찾았음): 표 조각에 "표 제목 + 행 이름 + 열 제목"이 들어가 질문과 단어가 더 많이 겹침

### 2. 처음 v1에서 고친 것 (보완)
| 문제 | 원인 | 고친 방법 | 결과 |
|---|---|---|---|
| 행 이름이 빈 행 126개 (`- : 4,175; …`) | "2025. 5"가 "2025."와 "5"로 나뉘어, "5"가 숫자 열로 오인됨 | 붙어 있는 "연도. 월"을 한 단어로 합침 | 빈 행 126 → 30개 (남은 것은 차트를 표로 잘못 찾은 것) |
| 큰 제목이 옆 열에 붙음 (`숙박 횟수 = 5.0`, `모집단 빈도 = 32,457`) | "가장 가까운 큰 제목"을 붙이는 규칙 | 열 바로 위의 제목 우선 → 큰 제목은 "자기가 덮는 열들의 가운데"가 맞는 동안만 넓힘 → 같은 높이에 자기 제목이 있는 열·열 묶음 경계는 넘지 않음 | B022·B028·T006 정답으로 바뀜, D02 수출액 표 "4분기" 열 이름도 바로잡힘 |

### 3. 차트(벡터) 점수가 내려간 이유 — 실행마다 흔들림
- 차트 문항의 정답 숫자는 v0과 v1 모두 **본문에 그대로 남아 있습니다**(확인: 18문항 중 17문항, 나머지 1문항은 v0에서도 "모름").
- 그런데 차트 숫자는 `14.5 15.0 11.5 12.7 12.2`처럼 **연도 순서 정보 없이** 나열되어, 모델이 어느 숫자가 몇 년도인지 짐작합니다.
- 그 결과 같은 문항이 실행마다 맞았다 틀렸다 합니다. 예: C020·C023은 v0 정답 → v1(처음) 오답 → v1(보완) 정답, C021·C029는 그 반대.
- 즉 하락은 표 변경 때문이 아니라 **원래 운에 기대던 문항**이기 때문입니다. 차트는 **v2(Vision 캡션)**에서 "연도 = 값" 형태로 바꿔 해결합니다.

### 4. 남은 표 오답 5문항
| 문항 | 상태 |
|---|---|
| B002 (25~29세 경제활동참가율) | 표 조각에 `2026. 5 청년층 25~29세 = 76.9`가 정확히 있고 같은 쪽(9쪽)도 검색됐으나 "모름". 검색된 것이 이 표 조각이 아닌 같은 쪽 본문 조각이었을 가능성이 있음 (쪽 단위 Recall로는 구분 안 됨) → 6주차에 조각 단위로 확인 |
| B004 (정보통신업 취업자 수) | 열 제목이 `청년층 취업자 구성비`로 섞여 모델이 "모름" |
| B005 (3년제 이하 여자 졸업 소요기간) | "2년 6.3개월"처럼 두 칸에 걸친 값 → 재구성 한계 |
| B010·B023 | 정답 쪽이 검색되지 않음 → 6주차 하이브리드 검색·리랭커 대상 |

### 5. 정리
- v1의 핵심 성과: **표 +30%p, 전체 정답률 +6.3%p, 환각 0% 유지**
- 남은 과제: 차트(v2), 검색 순위(v3)
- 1~2문항 차이는 실행 간 흔들림일 수 있으므로, 판단은 유형별 큰 변화를 기준으로 합니다.